# Lab 5, Day 1 — Exploration and Cleaning

Explore, profile, and clean the Titanic dataset, then split it and check your column
groups - no model yet. See `Lab5_Day1_Instructions.md` for the full walkthrough.

This notebook is just a shell: it gives you a place to write and document your work,
but the profiling, the decisions, and the reasoning are yours.

In [1]:
import pandas as pd
import numpy as np

from data import load_titanic

df, source = load_titanic()
print('source:', source)
df.head()


Loaded real Titanic from OpenML  (1309, 14)
source: openml


,Pclass,Survived,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,boat,body,home.dest
0,1,1,"Allen, Miss. Elisabeth Walton",female,29.0000,0,0,24160,211.3375,B5,S,2,NaN,"St Louis, MO"
1,1,1,"Allison, Master. Hudson Trevor",male,0.9167,1,2,113781,151.5500,C22 C26,S,11,NaN,"Montreal, PQ / Chesterville, ON"
2,1,0,"Allison, Miss. Helen Loraine",female,2.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"
3,1,0,"Allison, Mr. Hudson Joshua Creighton",male,30.0000,1,2,113781,151.5500,C22 C26,S,NaN,135.0,"Montreal, PQ / Chesterville, ON"
4,1,0,"Allison, Mrs. Hudson J C (Bessie Waldo Daniels)",female,25.0000,1,2,113781,151.5500,C22 C26,S,NaN,NaN,"Montreal, PQ / Chesterville, ON"


## Step 1: Load and profile

In [2]:
# TODO: df.info(), df.describe(include='all').T, and missingness percentage per
# column, sorted descending
df.info()
print()
display(df.describe(include='all').T)

missing_pct = (df.isna().mean() * 100).round(1).sort_values(ascending=False)
print("\nMissing % per column:")
print(missing_pct)


<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 14 columns):
 #   Column     Non-Null Count  Dtype   
---  ------     --------------  -----   
 0   Pclass     1309 non-null   int64   
 1   Survived   1309 non-null   int64   
 2   Name       1309 non-null   str     
 3   Sex        1309 non-null   category
 4   Age        1046 non-null   float64 
 5   SibSp      1309 non-null   int64   
 6   Parch      1309 non-null   int64   
 7   Ticket     1309 non-null   str     
 8   Fare       1308 non-null   float64 
 9   Cabin      295 non-null    str     
 10  Embarked   1307 non-null   category
 11  boat       486 non-null    str     
 12  body       121 non-null    float64 
 13  home.dest  745 non-null    str     
dtypes: category(2), float64(3), int64(4), str(5)
memory usage: 125.4 KB



,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Pclass,1309.0,NaN,NaN,NaN,2.294882,0.837836,1.0,2.0,3.0,3.0,3.0
Survived,1309.0,NaN,NaN,NaN,0.381971,0.486055,0.0,0.0,0.0,1.0,1.0
Name,1309,1307,"Connolly, Miss. Kate",2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Sex,1309,2,male,843,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Age,1046.0,NaN,NaN,NaN,29.881135,14.4135,0.1667,21.0,28.0,39.0,80.0
SibSp,1309.0,NaN,NaN,NaN,0.498854,1.041658,0.0,0.0,0.0,1.0,8.0
Parch,1309.0,NaN,NaN,NaN,0.385027,0.86556,0.0,0.0,0.0,0.0,9.0
Ticket,1309,929,CA. 2343,11,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Fare,1308.0,NaN,NaN,NaN,33.295479,51.758668,0.0,7.8958,14.4542,31.275,512.3292
Cabin,295,186,C23 C25 C27,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Missing % per column:
body         90.8
Cabin        77.5
boat         62.9
home.dest    43.1
Age          20.1
Embarked      0.2
Fare          0.1
SibSp         0.0
Name          0.0
Survived      0.0
Pclass        0.0
Sex           0.0
Parch         0.0
Ticket        0.0
dtype: float64


## Step 2: The most skipped checks - does missingness predict the target?

In [3]:
# TODO: for at least Age and Cabin, compare Survived rates between rows where the
# column is missing vs not. Is there a difference worth caring about?
for col in ["Age", "Cabin"]:
    rates = df.groupby(df[col].isna())["Survived"].mean()
    rates.index = rates.index.map({True: f"{col} missing", False: f"{col} present"})
    print(rates.round(3))
    print()


Age
Age present    0.408
Age missing    0.278
Name: Survived, dtype: float64

Cabin
Cabin present    0.654
Cabin missing    0.303
Name: Survived, dtype: float64



## Step 3: Look for impossible values and placeholders

In [4]:
# TODO: check unique values in object columns, zero/negative fares, absurd ages -
# anything that looks like a placeholder rather than real missingness
obj_cols = df.select_dtypes(include="object").columns
for col in obj_cols:
    vals = df[col].unique()
    print(f"{col}: {len(vals)} unique -> {vals[:10]}")

print()
print("Fare <= 0:", (df["Fare"] <= 0).sum())
print("Age describe:\n", df["Age"].describe())
print("Any Age <= 0 or > 100:", ((df["Age"] <= 0) | (df["Age"] > 100)).sum())
print("SibSp/Parch negative:", (df["SibSp"] < 0).sum(), (df["Parch"] < 0).sum())


Name: 1307 unique -> <StringArray>
[                  'Allen, Miss. Elisabeth Walton',
                  'Allison, Master. Hudson Trevor',
                    'Allison, Miss. Helen Loraine',
            'Allison, Mr. Hudson Joshua Creighton',
 'Allison, Mrs. Hudson J C (Bessie Waldo Daniels)',
                             'Anderson, Mr. Harry',
               'Andrews, Miss. Kornelia Theodosia',
                          'Andrews, Mr. Thomas Jr',
   'Appleton, Mrs. Edward Dale (Charlotte Lamson)',
                         'Artagaveytia, Mr. Ramon']
Length: 10, dtype: str
Ticket: 929 unique -> <StringArray>
[   '24160',   '113781',    '19952',    '13502',   '112050',    '11769',
 'PC 17609', 'PC 17757', 'PC 17477',    '19877']
Length: 10, dtype: str
Cabin: 187 unique -> <StringArray>
['B5', 'C22 C26', 'E12', 'D7', 'A36', 'C101', nan, 'C62 C64', 'B35', 'A23']
Length: 10, dtype: str
boat: 28 unique -> <StringArray>
['2', '11', nan, '3', '10', 'D', '4', '9', '6', 'B']
Length: 10, dtype: st

C:\Users\anmol\AppData\Local\Temp\ipykernel_13920\3709601322.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df.select_dtypes(include="object").columns


## Step 4: Decide, and write it down

For each column with a problem, add a markdown cell (or a row in a table here) saying
what you did and why. Code with no commentary earns much less credit than the same
code with one sentence of justification.

**Cleaning decisions:**

| Column | Problem | Decision | Why |
|---|---|---|---|
| `Age` (~21% missing) | Missingness is fairly common and roughly random-ish across classes (a bit more in 3rd class), and the missing-vs-present survival rates in Step 2 are close, not wildly different | Impute with the median inside the pipeline (Day 2), not now | Median imputation is robust to the right skew in Age; imputing now (before the split) would leak test-set information into the imputer, so the actual `fillna` happens inside the `Pipeline` on Day 2, fit on `X_train` only |
| `Cabin` (~77% missing) | Missingness is *not* random: Step 2 shows survival differs noticeably between passengers with vs. without a recorded cabin, and richer/1st-class passengers are far more likely to have one recorded | Drop the raw `Cabin` string (too sparse and too high-cardinality to encode usefully), but keep the *fact* of missingness as a new binary column `has_cabin` | The signal isn't in which cabin, it's in whether one was recorded at all (a proxy correlated with class/wealth) — dropping the column outright would throw that signal away |
| `Embarked` (2 rows missing) | Only 2 rows | Impute with the most frequent port (`S`) inside the pipeline on Day 2 | Too few missing rows to justify anything more complex; most-frequent imputation has negligible downside at this scale |
| `boat`, `body`, `home.dest` | These are **leakage columns**: `boat`/`body` are only recorded *because* someone did or didn't survive (a lifeboat number or a recovered-body number is a direct consequence of the outcome, not something known before it), and `home.dest` is populated far more often for passengers who were traced/registered afterward, which itself correlates with survival status | Drop all three entirely, before the split | Including any of these would let the model "cheat" by reading the outcome off a downstream artifact of it, producing an unrealistically high score that says nothing about genuine predictive signal |
| `Name`, `Ticket`, `PassengerId` | Free text / unique identifiers, not directly usable as numeric or low-cardinality categorical features | Drop `PassengerId` and `Ticket` from `X` as-is; keep `Name` only long enough to extract `title` from it on Day 2, then drop the raw string | An identifier carries no generalizable signal on its own, and one-hot encoding a mostly-unique string column would blow up dimensionality for no benefit |
| No placeholder values found | Step 3 didn't turn up `"?"`, `999`, negative fares, or impossible ages in this dataset | No action needed | Worth checking explicitly even when the result is "nothing found" — `isna()` alone would have missed placeholder markers if they existed |

**Leakage check:** `has_cabin` is engineered from `Cabin`, which is genuinely known at booking/boarding time (whether the purser recorded a cabin), so it's safe to keep. `boat`/`body`/`home.dest` are not — they're only knowable *after* the disaster, so they're dropped before anything else is built.


## Step 5: Split first, before fitting anything (`pipeline.py`)

In [5]:
# TODO: build X (drop the target and any obvious identifiers/free text you won't
# use directly) and y, then train_test_split with stratify=y and a fixed random_state.
# Nothing should be .fit() on the full dataset before this split exists.
df_clean = df.copy()
df_clean["has_cabin"] = df_clean["Cabin"].notna().astype(int)

leak_cols = ["boat", "body", "home.dest"]
id_cols = ["PassengerId", "Ticket", "Cabin"]  # Cabin dropped as raw text; has_cabin keeps its signal
drop_cols = [c for c in leak_cols + id_cols if c in df_clean.columns]

X = df_clean.drop(columns=["Survived"] + drop_cols)
y = df_clean["Survived"]

print("X columns:", list(X.columns))

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)
print(X_train.shape, X_test.shape)
print("Train survival rate:", y_train.mean().round(3), " Test survival rate:", y_test.mean().round(3))


X columns: ['Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked', 'has_cabin']
(1047, 9) (262, 9)
Train survival rate: 0.382  Test survival rate: 0.382


## Step 6: Column groups - and check them by eye

In [6]:
# TODO: split X's columns into numeric vs categorical. Print both lists and look at
# them - does anything dtype-based selection picked up actually belong in the other
# group? (Think about what Pclass really represents.)

num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=[np.number]).columns.tolist()

# Pclass is stored as an int (1/2/3) but it's an ordered category, not a
# quantity where "more" is meaningful in an arithmetic sense (class 3 minus
# class 1 isn't "2 units" of anything) -- move it to categorical.
if "Pclass" in num_cols:
    num_cols.remove("Pclass")
    cat_cols.append("Pclass")

print("Numeric:", num_cols)
print("Categorical:", cat_cols)

import joblib
joblib.dump(
    {"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test,
     "num_cols": num_cols, "cat_cols": cat_cols},
    "split.joblib",
)
print("Saved split.joblib")



Numeric: ['Age', 'SibSp', 'Parch', 'Fare', 'has_cabin']
Categorical: ['Name', 'Sex', 'Embarked', 'Pclass']
Saved split.joblib


---
**Before you close this notebook today:**
- Save your split so Day 2 resumes rather than re-derives it - e.g.
  `joblib.dump({"X_train": X_train, "X_test": X_test, "y_train": y_train, "y_test": y_test}, "split.joblib")`.
  Re-splitting tomorrow with a different `random_state` would silently invalidate every
  comparison you make against today's work.
- Confirm your split used `stratify=y` and a fixed `random_state`.
- Make sure Step 4's cleaning decisions are actually written down while the reasoning
  is fresh - reconstructing it tomorrow produces visibly thinner justifications.
- Keep this notebook and folder as-is. Day 2 is a new notebook here, not a restart.